In [4]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
tf.get_logger().setLevel("ERROR")

import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ipywidgets import interact, FloatSlider

from sklearn.datasets import load_breast_cancer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Seed 101 for unique graph values
SEED = 101
tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)
plt.style.use("seaborn-v0_8-whitegrid")
%matplotlib inline

print("Setup complete. You can start the experiment.")

Setup complete. You can start the experiment.


In [5]:
# Load Breast Cancer dataset and prepare Train/Val/Test sets

cancer = load_breast_cancer(as_frame=True)
X = cancer.data
y = cancer.target

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y,
    test_size=0.15,
    random_state=101,
    stratify=y,
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.1765,
    random_state=101,
    stratify=y_train_full,
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Train shape:", X_train_scaled.shape)
print("Validation shape:", X_val_scaled.shape)
print("Test shape:", X_test_scaled.shape)

Train shape: (397, 30)
Validation shape: (86, 30)
Test shape: (86, 30)


In [6]:
# Build the Perceptron Model

perceptron_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_scaled.shape[1],)),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])

perceptron_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.005),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

perceptron_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 1)              │            31 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 31 (124.00 B)

 Trainable params: 31 (124.00 B)

 Non-trainable params: 0 (0.00 B)

In [10]:
# Train the Perceptron with Early Stopping

perceptron_history = perceptron_model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=30,
    batch_size=32,
    verbose=0,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)],
)

print(f"Perceptron training finished after {len(perceptron_history.history['loss'])} epochs.")

Perceptron training finished after 30 epochs.


In [11]:
# Evaluate Perceptron on Test Set

perceptron_test_probs = perceptron_model.predict(X_test_scaled, verbose=0).ravel()
perceptron_test_pred = (perceptron_test_probs >= 0.5).astype(int)

perceptron_accuracy = accuracy_score(y_test, perceptron_test_pred)
print(f"Perceptron Test Accuracy : {perceptron_accuracy:.4f}")
print(classification_report(y_test, perceptron_test_pred, target_names=cancer.target_names))

Perceptron Test Accuracy : 0.9767
              precision    recall  f1-score   support

   malignant       1.00      0.94      0.97        32
      benign       0.96      1.00      0.98        54

    accuracy                           0.98        86
   macro avg       0.98      0.97      0.97        86
weighted avg       0.98      0.98      0.98        86



In [12]:
# Build the MLP Model

mlp_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_scaled.shape[1],)),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])

mlp_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.002),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

mlp_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_1 (Dense)                 │ (None, 32)             │           992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,537 (6.00 KB)

 Trainable params: 1,537 (6.00 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
# Train MLP with Early Stopping

mlp_history = mlp_model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=100,
    batch_size=32,
    verbose=0,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True)],
)

print(f"MLP training finished after {len(mlp_history.history['loss'])} epochs.")

MLP training finished after 29 epochs.


In [15]:
# Evaluate MLP on Test Set

mlp_test_probs = mlp_model.predict(X_test_scaled, verbose=0).ravel()
mlp_test_pred = (mlp_test_probs >= 0.5).astype(int)

mlp_accuracy = accuracy_score(y_test, mlp_test_pred)
print(f"MLP Test Accuracy : {mlp_accuracy:.4f}")
print(classification_report(y_test, mlp_test_pred, target_names=cancer.target_names))

MLP Test Accuracy : 0.9651
              precision    recall  f1-score   support

   malignant       0.97      0.94      0.95        32
      benign       0.96      0.98      0.97        54

    accuracy                           0.97        86
   macro avg       0.97      0.96      0.96        86
weighted avg       0.97      0.97      0.96        86



In [16]:
# Interactive Learning Curves: Accuracy

fig = make_subplots(rows=1, cols=2, subplot_titles=("Perceptron Accuracy", "MLP Accuracy"))

fig.add_trace(go.Scatter(y=perceptron_history.history["accuracy"], mode="lines", name="Train",
                          line=dict(color="#2a9d8f")), row=1, col=1)
fig.add_trace(go.Scatter(y=perceptron_history.history["val_accuracy"], mode="lines", name="Validation",
                          line=dict(color="#e76f51")), row=1, col=1)

fig.add_trace(go.Scatter(y=mlp_history.history["accuracy"], mode="lines", name="Train",
                          line=dict(color="#2a9d8f"), showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(y=mlp_history.history["val_accuracy"], mode="lines", name="Validation",
                          line=dict(color="#e76f51"), showlegend=False), row=1, col=2)

fig.update_xaxes(title_text="Epoch")
fig.update_yaxes(title_text="Accuracy")
fig.update_layout(title="Training vs Validation Accuracy", height=450, width=950, template="plotly_white")

fig.show()

In [18]:
# Interactive Learning Curves: Loss

fig = make_subplots(rows=1, cols=2, subplot_titles=("Perceptron Loss", "MLP Loss"))

fig.add_trace(go.Scatter(y=perceptron_history.history["loss"], mode="lines", name="Train",
                          line=dict(color="#2a9d8f")), row=1, col=1)
fig.add_trace(go.Scatter(y=perceptron_history.history["val_loss"], mode="lines", name="Validation",
                          line=dict(color="#e76f51")), row=1, col=1)

fig.add_trace(go.Scatter(y=mlp_history.history["loss"], mode="lines", name="Train",
                          line=dict(color="#2a9d8f"), showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(y=mlp_history.history["val_loss"], mode="lines", name="Validation",
                          line=dict(color="#e76f51"), showlegend=False), row=1, col=2)

fig.update_xaxes(title_text="Epoch")
fig.update_yaxes(title_text="Binary Cross-Entropy Loss")
fig.update_layout(title="Training vs Validation Loss", height=450, width=950, template="plotly_white")

fig.show()

In [19]:
# Interactive Confusion Matrices

perceptron_cm = confusion_matrix(y_test, perceptron_test_pred)
mlp_cm = confusion_matrix(y_test, mlp_test_pred)
labels = list(cancer.target_names)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Perceptron", "MLP"))

fig.add_trace(go.Heatmap(z=perceptron_cm.tolist(), x=labels, y=labels, colorscale="Blues",
                          text=perceptron_cm.tolist(), texttemplate="%{text}", showscale=False), row=1, col=1)
fig.add_trace(go.Heatmap(z=mlp_cm.tolist(), x=labels, y=labels, colorscale="Blues",
                          text=mlp_cm.tolist(), texttemplate="%{text}", showscale=False), row=1, col=2)

fig.update_xaxes(title_text="Predicted Label")
fig.update_yaxes(title_text="True Label", autorange="reversed")
fig.update_layout(title="Confusion Matrix Comparison", height=450, width=950, template="plotly_white")

fig.show()

In [21]:
# Interactive ROC Curve Comparison

fpr_p, tpr_p, _ = roc_curve(y_test, perceptron_test_probs)
fpr_m, tpr_m, _ = roc_curve(y_test, mlp_test_probs)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=fpr_p, y=tpr_p, mode="lines", name=f"Perceptron (AUC = {roc_auc_score(y_test, perceptron_test_probs):.3f})",
    line=dict(color="#2a9d8f")
))
fig.add_trace(go.Scatter(
    x=fpr_m, y=tpr_m, mode="lines", name=f"MLP (AUC = {roc_auc_score(y_test, mlp_test_probs):.3f})",
    line=dict(color="#e76f51")
))
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode="lines", name="Random Guess",
    line=dict(color="gray", dash="dash")
))

fig.update_layout(
    title="ROC Curve Comparison",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    height=500, width=650,
    template="plotly_white"
)

fig.show()

In [22]:
# Interactive Threshold Explorer for the MLP

def explore_threshold(threshold=0.50):
    y_pred_t = (mlp_test_probs >= threshold).astype(int)

    acc = accuracy_score(y_test, y_pred_t)
    prec = precision_score(y_test, y_pred_t, zero_division=0)
    rec = recall_score(y_test, y_pred_t, zero_division=0)
    f1 = f1_score(y_test, y_pred_t, zero_division=0)
    cm = confusion_matrix(y_test, y_pred_t)

    print(f"Threshold : {threshold:.2f}")
    print(f"Accuracy  : {acc:.4f}")
    print(f"Precision : {prec:.4f}")
    print(f"Recall    : {rec:.4f}")
    print(f"F1-score  : {f1:.4f}")

    fig = go.Figure(data=go.Heatmap(
        z=cm.tolist(), x=labels, y=labels, colorscale="Blues",
        text=cm.tolist(), texttemplate="%{text}", showscale=False
    ))
    fig.update_xaxes(title_text="Predicted Label")
    fig.update_yaxes(title_text="True Label", autorange="reversed")
    fig.update_layout(
        title=f"MLP Confusion Matrix at Threshold = {threshold:.2f}",
        height=420, width=420, template="plotly_white"
    )
    fig.show()

interact(
    explore_threshold,
    threshold=FloatSlider(value=0.50, min=0.05, max=0.95, step=0.05,
                           description="Threshold", continuous_update=False)
);

interactive(children=(FloatSlider(value=0.5, continuous_update=False, description='Threshold', max=0.95, min=0…